In [2]:
import os
import h5py
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.cluster import MiniBatchKMeans

import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [3]:
os.environ['OPENBLAS_NUM_THREADS']= "64"
os.environ['OMP_NUM_THREADS']= "64"

In [4]:
def predict_all(num_clusters, downsample_file, label_file, slide_file): 
    cancer_types = ['paad','coad','luad','esca','stad','read']
    mbk = MiniBatchKMeans(n_clusters=num_clusters,random_state = 0) #customize number of clusters
    downsample = pd.read_pickle(downsample_file) #add file name 
    mbk.fit(downsample.iloc[:,:1024])

    predictions_and_coordinates = {cancer_type:{} for cancer_type in cancer_types}

    np.random.seed(12344412)
    for cancer_type in cancer_types:
        cond = [a for a in os.listdir('/data/vshir/tcga/{}_40'.format(cancer_type)) if '40x_' in a][0]
        
        feature_dir = '/data/vshir/tcga/{}_40/{}/features_uni_v1'.format(cancer_type,cond)
        for file in tqdm(os.listdir(feature_dir)):
            if file.endswith(".h5"):
                file_path = os.path.join(feature_dir, file)
                h5f = h5py.File(file_path, "r")
                
                # with  as h5f:
                #     features.setdefault(cond, {})[file[:-3]] = h5f["features"][:]
                coord_dir = '/data/vshir/tcga/{}_40/{}/patches'.format(cancer_type, cond)
                coord_h5f = h5py.File(os.path.join(coord_dir, os.path.join(file[:-3]+'_patches.h5')), "r")
    
                features = h5f["features"][:]
                coordinates = coord_h5f["coords"][:]
                predictions_and_coordinates[cancer_type][file[:-3]] = (mbk.predict(features.astype(np.float64)),coordinates)
                h5f.close()
                coord_h5f.close()
    
    labelled_data = pd.concat(({cancer_type : pd.concat({slide_id: pd.DataFrame({'label': v[0], 'X':v[1][:,0], 'Y':v[1][:,1]}) for slide_id,v in predictions_and_coordinates[cancer_type].items()}) for cancer_type in cancer_types}))
    labelled_data['label'] = labelled_data['label'].astype('category')
    labelled_data.to_pickle(label_file) #change this 
    
    test_slide_ids = labelled_data.loc[cancer_type].sample(10,random_state = 4).reset_index()['level_0'].unique() #input cancer type 
    all_slide_ids = pd.concat({k: pd.Series(v.keys()) for k,v in predictions_and_coordinates.items()})
    all_slide_ids.to_pickle(slide_file) #customize file name 

    for slide_id in test_slide_ids:
        sns.scatterplot(data = labelled_data.loc[cancer_type].loc[slide_id], x = 'X', y = 'Y', hue = 'label',s = 2,legend = False,palette = 'bright') #change input 
        plt.show()
    

In [ ]:
predict_all(40, 'pkl/GI_40x256_120k30x_mar19_2025.pkl', 'pkl/GI_cluster40_labelled_all_02_18_26.pkl', 'pkl/GI_allslide_IDs_02_18_26.pkl')

In [ ]:
predict_all(60, 'pkl/GI_40x256_120k30x_mar19_2025.pkl', 'pkl/GI_cluster60_labelled_all_02_18_26.pkl', 'pkl/GI_cluster40_labelled_all_02_18_26.pkl